In [1]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [2]:
from glob import glob

In [3]:
splits = ['train', 'dev', 'test']
lang = 'indo'
dataset_types = [x.split('/')[-1] for x in glob(f'dataset/*')]
dataset_types = [i for i in dataset_types if 'hoasa_hotel' not in i]
dataset_types = [i for i in dataset_types if 'aug' not in i]
dataset_types = ['hotel_reviews', 'hoasa']

dataset_folders = []
for dataset_type in dataset_types:
    dataset_folders.extend(glob(f'dataset/{dataset_type}/{lang}/*'))
dataset_folders = [x.split('/')[-1] for x in dataset_folders]
dataset_folders = list(set(dataset_folders))
dataset_folders = [i for i in dataset_folders if 'raw' not in i and 'augment' not in i]
print(f'Dataset types: {dataset_types}')
print(f'Dataset folders: {dataset_folders}')

Dataset types: ['hotel_reviews', 'hoasa']
Dataset folders: ['mvp', 'indolegoabsa_multitask', 'mvp_aos', 'gas', 'legoabsa_multitask']


In [4]:
from copy import deepcopy

In [5]:
new_dataset_type = 'hoasa_hotel'
for dataset_folder in dataset_folders:
	instance_id = 0
	for split in splits:
		new_dataset = []

		for dataset_type in dataset_types:
			with open(f'dataset/{dataset_type}/{lang}/{dataset_folder}/{split}.json') as f:
				temp_dataset = json.load(f)
			for i in range(len(temp_dataset)):
				temp_dataset[i]['dataset_type'] = dataset_type
			new_dataset.extend(deepcopy(temp_dataset))
			
		
		for i in range(len(new_dataset)):
			# Reset instance id
			new_dataset[i]['instance_id'] = instance_id
			instance_id += 1
	
		# Write the new dataset
		os.makedirs(f'dataset/{new_dataset_type}/{lang}/{dataset_folder}', exist_ok=True)
		with open(f'dataset/{new_dataset_type}/{lang}/{dataset_folder}/{split}.json', 'w') as f:
			json.dump(new_dataset, f, ensure_ascii=False, indent=4)